# BISTRO-XAI: Attention Analysis & Feature Selection Walkthrough

한국 CPI(소비자물가 전년동월비) 예측을 위한 BISTRO 모델의 XAI 분석 파이프라인.

## 구조
1. **데이터 확인** — 일별 거시경제 패널 (29개 변수)
2. **Stage 1 결과** — 전체 변수 Attention 스크리닝
3. **Stage 2 결과** — 선택된 10개 변수 재추론 + Attention Map
4. **Ablation Study** — 변수 제거 실험 (ΔRMSE)
5. **2×2 Diagnostic** — Attention vs Ablation 교차 진단

> 이 노트북은 **사전 계산된 결과**를 로딩합니다. 모델 재실행은 Section 6 참조.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 결과 파일 경로
DATA_DIR = "data"
s1 = np.load(f"{DATA_DIR}/stage1_screening.npz", allow_pickle=True)
s2 = np.load(f"{DATA_DIR}/real_inference_results.npz", allow_pickle=True)
ab = np.load(f"{DATA_DIR}/ablation_results.npz", allow_pickle=True)

print(f"Data mode: {s1['data_mode']}")
print(f"Stage 1: {s1['n_variates']} variates, attention shape {s1['attn_arrays'].shape}")
print(f"Stage 2: {s2['n_variates']} variates, attention shape {s2['attn_arrays'].shape}")
print(f"Ablation: {len(ab['abl_vars'])} variables tested")

---
## 1. 데이터 확인

In [ ]:
# 일별 패널 로딩
panel = pd.read_csv(f"{DATA_DIR}/macro_panel_daily.csv", index_col=0, parse_dates=True)
freq_df = pd.read_csv(f"{DATA_DIR}/variable_freq.csv")
freq_map = dict(zip(freq_df["variable"], freq_df["original_freq"]))

print(f"패널: {panel.shape[0]:,}일 × {panel.shape[1]}변수")
print(f"기간: {panel.index[0].strftime('%Y-%m-%d')} ~ {panel.index[-1].strftime('%Y-%m-%d')}")
print(f"\nDaily 변수 ({sum(v=='daily' for v in freq_map.values())}개):")
for v, f in freq_map.items():
    if f == "daily": print(f"  {v}")
print(f"\nMonthly 변수 ({sum(v=='monthly' for v in freq_map.values())}개):")
for v, f in freq_map.items():
    if f == "monthly": print(f"  {v}")

In [ ]:
# CPI 시계열 미리보기
cpi = panel["CPI_KR_YoY"].resample("M").last().dropna()
fig = go.Figure()
fig.add_trace(go.Scatter(x=cpi.index, y=cpi.values, name="CPI_KR_YoY"))
fig.update_layout(title="한국 CPI 전년동월비 (%)", height=350,
                  xaxis_title="Date", yaxis_title="YoY %")
fig.show()

---
## 2. Stage 1: 전체 변수 Attention 스크리닝 (29개)

In [ ]:
# Stage 1 Attention 랭킹
s1_vars = s1["s1_ranking_vars"].tolist()
s1_attn = s1["s1_ranking_attn"]
s1_self = float(s1["s1_self_attn"])
s1_uniform = float(s1["s1_uniform_share"])
s2_selected = s2["s2_selected_vars"].tolist()

print(f"Self-attention (CPI→CPI): {s1_self:.1%}")
print(f"Cross-attention budget:   {1-s1_self:.1%}")
print(f"Uniform share baseline:   {s1_uniform:.2%}")
print(f"Selected for Stage 2:     {len(s2_selected)}개")
print(f"\n{'Rank':<5} {'Variable':<18} {'Attention':>10} {'vs Uniform':>12} {'Selected':>10}")
print("-" * 60)
for i, (v, a) in enumerate(zip(s1_vars, s1_attn), 1):
    sel = "✅" if v in s2_selected else "  "
    print(f"{i:<5} {v:<18} {a:>9.2%} {a/s1_uniform:>10.1f}×   {sel}")

In [ ]:
# Stage 1 Attention 랭킹 차트
colors = ["#1A6FD4" if v in s2_selected else "#CCCCCC" for v in s1_vars]
cumsum = np.cumsum(s1_attn)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(
    x=s1_vars, y=s1_attn, marker_color=colors,
    text=[f"{a:.1%}" for a in s1_attn], textposition="outside",
    name="Attention",
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=s1_vars, y=cumsum, mode="lines+markers",
    line=dict(color="#D62728", width=2.5), name="Cumulative",
), secondary_y=True)
fig.add_hline(y=s1_uniform, line_dash="dot", line_color="#999",
              annotation_text=f"Uniform: {s1_uniform:.2%}", secondary_y=False)
fig.update_layout(
    title="Stage 1: Attention-Based Feature Screening (28 covariates)",
    xaxis=dict(tickangle=-40), yaxis=dict(tickformat=".1%"),
    yaxis2=dict(tickformat=".0%", range=[0, 1.05]),
    height=500, legend=dict(x=0.01, y=0.99),
)
fig.show()

---
## 3. Stage 2: 선택된 10개 변수 재추론

In [ ]:
# 예측 결과
dates = s2["forecast_date"]
forecast_df = pd.DataFrame({
    "Date": dates,
    "BISTRO": s2["forecast_med"],
    "CI_Lo": s2["forecast_ci_lo"],
    "CI_Hi": s2["forecast_ci_hi"],
    "AR(1)": s2["forecast_ar1"],
    "Actual": s2["forecast_actual"],
})
print(forecast_df.to_string(index=False, float_format="{:.3f}".format))

# RMSE / MAE
residuals = forecast_df["BISTRO"].values - forecast_df["Actual"].values
rmse = np.sqrt(np.mean(residuals**2))
mae = np.mean(np.abs(residuals))
print(f"\nBISTRO  RMSE: {rmse:.3f}pp  MAE: {mae:.3f}pp")

res_ar1 = forecast_df["AR(1)"].values - forecast_df["Actual"].values
print(f"AR(1)   RMSE: {np.sqrt(np.mean(res_ar1**2)):.3f}pp  MAE: {np.mean(np.abs(res_ar1)):.3f}pp")

In [ ]:
# 예측 vs 실제 차트
hist_dates = s2["history_date"]
hist_cpi = s2["history_cpi"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=hist_dates, y=hist_cpi, name="History",
                         line=dict(color="#333", width=1.5)))
fig.add_trace(go.Scatter(x=dates, y=forecast_df["Actual"], name="Actual",
                         line=dict(color="#333", width=2.5)))
fig.add_trace(go.Scatter(x=dates, y=forecast_df["BISTRO"], name="BISTRO",
                         line=dict(color="#1A6FD4", width=2.5)))
fig.add_trace(go.Scatter(
    x=list(dates) + list(dates)[::-1],
    y=list(forecast_df["CI_Hi"]) + list(forecast_df["CI_Lo"])[::-1],
    fill="toself", fillcolor="rgba(26,111,212,0.15)",
    line=dict(width=0), name="90% CI", showlegend=True,
))
fig.add_trace(go.Scatter(x=dates, y=forecast_df["AR(1)"], name="AR(1)",
                         line=dict(color="#999", dash="dot", width=1.5)))
fig.update_layout(title="BISTRO vs Actual: Korean CPI YoY (2023)",
                  height=450, yaxis_title="CPI YoY %")
fig.show()

---
## 4. Cross-Variate Attention Matrix (Stage 2)

In [ ]:
# Attention matrix 계산 (token → variate 집계)
# 대시보드와 동일하게 마지막 레이어(Layer 12)만 사용
# (residual connection 구조상 최종 출력에 직접 반영되는 레이어)
variates = s2["variates"].tolist()
n_var = int(s2["n_variates"])
ctx_p = int(s2["ctx_patches"])
attn_arrays = s2["attn_arrays"]  # (12 layers, T, T)

# 마지막 레이어만 사용 (Layer 12)
last_layer_attn = attn_arrays[-1]  # (T, T)

# token → variate 블록 평균
cross = np.zeros((n_var, n_var))
for qi in range(n_var):
    q_start, q_end = qi * ctx_p, (qi + 1) * ctx_p
    row_sum = 0.0
    for ki in range(n_var):
        k_start, k_end = ki * ctx_p, (ki + 1) * ctx_p
        cross[qi, ki] = last_layer_attn[q_start:q_end, k_start:k_end].mean()
        row_sum += cross[qi, ki]
    if row_sum > 0:
        cross[qi] /= row_sum  # normalize rows to sum=1

cross_df = pd.DataFrame(cross, index=variates, columns=variates)
print(f"Cross-variate matrix: {cross_df.shape} (Layer 12 only)")
print(f"\nCPI→CPI (self-attention): {cross_df.loc['CPI_KR_YoY','CPI_KR_YoY']:.1%}")
print(f"\nCPI row (what CPI attends to):")
cpi_row = cross_df.loc["CPI_KR_YoY"].sort_values(ascending=False)
for v, a in cpi_row.items():
    print(f"  {v:<18} {a:.2%}")

In [ ]:
# Cross-Variate Heatmap
z_pct = [[f"{v*100:.1f}%" for v in row] for row in cross_df.values]

fig = go.Figure(data=go.Heatmap(
    z=cross_df.values, x=variates, y=variates,
    colorscale="YlOrRd", zmin=0, zmax=0.6,
    text=z_pct, texttemplate="%{text}",
    hovertemplate="Query: <b>%{y}</b><br>Key: <b>%{x}</b><br>%{z:.1%}<extra></extra>",
    colorbar=dict(title="Attention", tickformat=".0%", dtick=0.1),
    xgap=1, ygap=1,
))
fig.add_shape(type="rect", x0=-0.5, x1=n_var-0.5, y0=-0.5, y1=0.5,
              line=dict(color="#3A7BD5", width=2.5))
fig.update_layout(
    title="Cross-Variate Attention Matrix (Stage 2, 11 variates)",
    xaxis=dict(title="Key", tickangle=-35),
    yaxis=dict(title="Query", autorange="reversed"),
    height=550, margin=dict(l=140, b=130),
)
fig.show()

---
## 5. Ablation Study — 변수 제거 실험

In [ ]:
# Ablation 결과
abl_vars = ab["abl_vars"].tolist()
abl_delta = ab["abl_delta_rmse"]
abl_rmse = ab["abl_rmse"]
baseline_rmse = float(ab["baseline_rmse"])

# Attention 랭킹
attn_ranking = ab["attn_ranking"].tolist()
attn_values = ab["attn_values"]

print(f"Baseline RMSE (10 covariates): {baseline_rmse:.4f}pp")
print(f"\n{'Variable':<18} {'RMSE':>8} {'ΔRMSE':>8} {'Attn Rank':>10} {'Attn':>8}")
print("-" * 58)

# ΔRMSE 내림차순
order = np.argsort(-abl_delta)
for i in order:
    v = abl_vars[i]
    attn_rank = attn_ranking.index(v) + 1 if v in attn_ranking else "—"
    attn_val = attn_values[attn_ranking.index(v)] if v in attn_ranking else 0
    marker = "⬆" if abl_delta[i] > 0 else "⬇"
    print(f"{v:<18} {abl_rmse[i]:>7.4f} {marker}{abs(abl_delta[i]):>7.4f} {attn_rank:>10} {attn_val:>7.2%}")

n_positive = int(np.sum(abl_delta > 0))
print(f"\nΔRMSE > 0 (기여): {n_positive}개 / {len(abl_vars)}개")
print(f"Optimal covariates: {n_positive}개")

In [ ]:
# ΔRMSE 막대 차트
sort_idx = np.argsort(abl_delta)
sorted_vars = [abl_vars[i] for i in sort_idx]
sorted_delta = abl_delta[sort_idx]
colors = ["#E24B4A" if d > 0 else "#4C9BE8" for d in sorted_delta]

fig = go.Figure(go.Bar(
    x=sorted_delta, y=sorted_vars, orientation="h",
    marker_color=colors,
    text=[f"{d:+.4f}" for d in sorted_delta], textposition="outside",
))
fig.add_vline(x=0, line_color="#333", line_width=1.5)
fig.update_layout(
    title="Ablation ΔRMSE (제거 시 RMSE 변화량)",
    xaxis_title="ΔRMSE (pp) — 양수 = 제거하면 성능 저하 = 기여",
    height=450, margin=dict(l=140),
)
fig.show()

---
## 6. 2×2 Diagnostic: Attention vs Ablation

In [ ]:
# Attention score (normalized) + Ablation ΔRMSE
attn_norm = attn_values / attn_values.max()  # 0~1 정규화
delta_max = np.abs(abl_delta).max()

# 변수별 매핑
rows = []
for v in abl_vars:
    ai = attn_ranking.index(v) if v in attn_ranking else -1
    a_norm = attn_norm[ai] if ai >= 0 else 0
    bi = abl_vars.index(v)
    d = abl_delta[bi]
    
    # 사분면 분류 (threshold: attention=0.5, delta=0)
    if a_norm >= 0.5 and d > 0:
        quad = "Confirmed Driver"
    elif a_norm >= 0.5 and d <= 0:
        quad = "Spurious Attention"
    elif a_norm < 0.5 and d > 0:
        quad = "Hidden Contributor"
    else:
        quad = "Irrelevant"
    
    rows.append({"Variable": v, "Attn_Norm": a_norm, "ΔRMSE": d, "Quadrant": quad})

diag_df = pd.DataFrame(rows)
print(diag_df.to_string(index=False, float_format="{:.4f}".format))

In [ ]:
# 2×2 Scatter
QUAD_COLORS = {
    "Confirmed Driver": "#E24B4A",
    "Spurious Attention": "#EF9F27",
    "Hidden Contributor": "#4C9BE8",
    "Irrelevant": "#AAAAAA",
}

fig = go.Figure()
for quad, color in QUAD_COLORS.items():
    mask = diag_df["Quadrant"] == quad
    sub = diag_df[mask]
    if len(sub) == 0:
        continue
    fig.add_trace(go.Scatter(
        x=sub["Attn_Norm"], y=sub["ΔRMSE"],
        mode="markers+text", name=quad,
        marker=dict(size=14, color=color, line=dict(width=1, color="#333")),
        text=sub["Variable"], textposition="top center", textfont=dict(size=10),
    ))

fig.add_vline(x=0.5, line_dash="dash", line_color="#666", line_width=1)
fig.add_hline(y=0.0, line_dash="dash", line_color="#666", line_width=1)

# 사분면 라벨
for label, x, y in [
    ("Confirmed Driver", 0.85, 0.9), ("Spurious Attention", 0.85, 0.1),
    ("Hidden Contributor", 0.15, 0.9), ("Irrelevant", 0.15, 0.1),
]:
    fig.add_annotation(x=x, y=y, xref="paper", yref="paper",
                       text=f"<b>{label}</b>", showarrow=False,
                       font=dict(size=12, color=QUAD_COLORS[label]),
                       opacity=0.6)

fig.update_layout(
    title="2×2 Diagnostic: Attention vs Ablation",
    xaxis_title="Attention Score (normalized)",
    yaxis_title="Ablation ΔRMSE (pp)",
    height=550,
)
fig.show()

### Key Findings

| 발견 | 설명 |
|------|------|
| **Attention ≠ Importance** | CNY_USD는 Attention 1위지만 ΔRMSE ≈ 0 (Spurious Attention) |
| **Hidden Contributor** | JP_Interbank3M은 Attention 중위권이지만 Ablation 1위 |
| **Foundation Model 특성** | Self-attention ~52%, 나머지는 거의 균등 배분 (LOTSA 사전학습 패턴) |
| **Instance Normalization** | 균일한 ±1σ CF 교란을 상쇄 → CF 대신 Ablation이 유효 |

---
## 7. Incremental 분석 — 변수 추가 순서별 RMSE

In [ ]:
# Incremental: attention 순서대로 변수를 1개씩 추가
inc_labels = ab["inc_labels"].tolist()
inc_rmse = ab["inc_rmse"]
inc_n = ab["inc_n_vars"]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=inc_n, y=inc_rmse, mode="lines+markers",
    line=dict(color="#1A6FD4", width=2.5),
    marker=dict(size=8),
    text=inc_labels, hovertemplate="%{text}<br>RMSE: %{y:.4f}<extra></extra>",
))
fig.add_hline(y=baseline_rmse, line_dash="dot", line_color="#E24B4A",
              annotation_text=f"Full model: {baseline_rmse:.4f}")

best_idx = np.argmin(inc_rmse)
fig.add_annotation(x=int(inc_n[best_idx]), y=float(inc_rmse[best_idx]),
                   text=f"Best: {inc_n[best_idx]} vars",
                   showarrow=True, arrowhead=2, ax=40, ay=-30)

fig.update_layout(
    title="Incremental RMSE: Attention 순서 변수 추가",
    xaxis_title="# Covariates", yaxis_title="RMSE (pp)",
    height=400,
)
fig.show()

---
## 8. (Optional) 전체 파이프라인 재실행

아래 코드는 `.venv-bistro` 환경 (Python 3.11 + uni2ts + torch)에서만 동작합니다.

```bash
# 1. 일별 데이터 생성
.venv-bistro/bin/python3 data_collector.py --daily

# 2. Stage 1 + Stage 2 추론
.venv-bistro/bin/python3 bistro_runner_30var.py --daily --top-k 10

# 3. Ablation Study
.venv-bistro/bin/python3 ablation_study.py

# 4. Streamlit 대시보드
.venv/bin/streamlit run app.py
```

결과 파일:
- `data/stage1_screening.npz` — Stage 1 전체 29변수 Attention
- `data/real_inference_results.npz` — Stage 2 선택 11변수 Attention + 예측
- `data/ablation_results.npz` — 변수 제거 실험 결과